In [1]:
import re
import numpy as np
import pandas as pd

In [2]:
def make_orders():
    rng = np.random.default_rng(101)
    n = 500
    df = pd.DataFrame({
        "order_id": [f"ORD{i:04d}" for i in range(1, n + 1)],
        "customer_id": rng.integers(100, 230, n),
        "city": rng.choice(["Delhi", "Mumbai", "Pune", "Jaipur", "Bengaluru"], n),
        "channel": rng.choice(["App", "Website", "Store"], n),
        "category": rng.choice(["Electronics", "Furniture", "Clothing", "Grocery"], n),
        "order_date": pd.Timestamp("2026-06-01") + pd.to_timedelta(rng.integers(0, 122, n), unit="D"),
        "quantity": rng.integers(1, 7, n),
        "unit_price": rng.integers(250, 22001, n).astype(float),
        "discount_pct": rng.choice([0.0, 5.0, 10.0, 15.0, 20.0], n),
        "shipping_fee": rng.choice([0.0, 49.0, 99.0, 149.0], n),
        "payment": rng.choice(["Prepaid", "COD"], n, p=[0.7, 0.3]),
        "status": rng.choice(["Delivered", "Returned", "Cancelled", "In transit"], n, p=[0.6, 0.15, 0.1, 0.15]),
        "delivery_days": rng.integers(1, 12, n).astype(float),
        "refund_amount": rng.choice([0.0, 0.0, 0.0, 250.0, 700.0, np.nan], n),
        "coupon_code": rng.choice([" vip05 ", "NEW20", "save10", "", None, "A1B20", "NEW00", "VIP75", "new20"], n),
    }, index=pd.Index(1000 + np.arange(n) * 3, name="row_id"))
    df.loc[df["status"].isin(["Cancelled", "In transit"]), "delivery_days"] = np.nan
    return df

orders = make_orders()

In [3]:
orders.head()

,order_id,customer_id,city,channel,category,order_date,quantity,unit_price,discount_pct,shipping_fee,payment,status,delivery_days,refund_amount,coupon_code
row_id,,,,,,,,,,,,,,,
1000,ORD0001,140,Bengaluru,Store,Electronics,2026-09-11,5,14802.0,20.0,49.0,COD,Delivered,1.0,700.0,save10
1003,ORD0002,222,Delhi,App,Electronics,2026-09-28,1,17871.0,20.0,0.0,Prepaid,Delivered,4.0,NaN,NaN
1006,ORD0003,190,Delhi,Store,Clothing,2026-07-17,5,3985.0,10.0,49.0,COD,Cancelled,NaN,0.0,NEW00
1009,ORD0004,146,Delhi,App,Furniture,2026-08-29,6,14406.0,15.0,0.0,Prepaid,Delivered,8.0,700.0,A1B20
1012,ORD0005,117,Jaipur,Website,Electronics,2026-07-21,6,20013.0,0.0,49.0,Prepaid,Delivered,11.0,0.0,VIP75


**Question 1**
Using loc, select delivered, prepaid orders from Delhi, Mumbai or Pune placed from 1 July through 30 September 2026, inclusive, with no recorded refund. Keep orders that are either Electronics with quantity at least 2, or Furniture with unit price at least ₹15,000.

Treat a missing refund as zero. Return order_id, order_date, city, category, quantity, unit_price and refund_amount, sorted by index ascending.

Sample output — 17 rows in the full result; first five shown.

In [8]:
delivered_users = orders.loc[
    orders['city'].isin(["Delhi", "Mumbai", "Pune"])&
    (orders['order_date']>'2026-07-01')&
    (orders['order_date']<'2026-09-30')

]

no_refunded_users = delivered_users.loc[
    delivered_users['refund_amount'].notnull()&
    (delivered_users['refund_amount']==0)]

keep_orders = no_refunded_users.loc[
    ((no_refunded_users['category']=="Electronics")&
    (no_refunded_users['quantity']>2))|
    ((no_refunded_users['category']=="Furniture")&
    (no_refunded_users['unit_price']>15000))
]
keep_orders.sort_values(by='row_id').head()

,order_id,customer_id,city,channel,category,order_date,quantity,unit_price,discount_pct,shipping_fee,payment,status,delivery_days,refund_amount,coupon_code
row_id,,,,,,,,,,,,,,,
1039,ORD0014,213,Pune,Website,Electronics,2026-09-08,5,14565.0,20.0,49.0,COD,Delivered,1.0,0.0,
1180,ORD0061,151,Mumbai,App,Electronics,2026-08-21,3,6347.0,5.0,0.0,COD,Returned,3.0,0.0,save10
1234,ORD0079,202,Delhi,Website,Furniture,2026-08-31,4,15605.0,0.0,149.0,Prepaid,Delivered,11.0,0.0,NEW20
1285,ORD0096,190,Delhi,Store,Electronics,2026-08-24,3,2111.0,5.0,0.0,COD,Delivered,9.0,0.0,NEW00
1315,ORD0106,159,Mumbai,App,Electronics,2026-08-31,3,7976.0,10.0,149.0,COD,Returned,3.0,0.0,NaN


In [4]:
# def cancelled_order_to_zero(price):
#     return 0
cancelled_orders = orders.loc[(orders['status']=="Cancelled")]
cancelled_orders['new_shipping_fee'] = 0
cancelled_orders['new_discount_pct'] = 0

refund_orders = cancelled_orders.loc[
    (cancelled_orders['delivery_days']>6)|
    (cancelled_orders['delivery_days'].isnull())|(cancelled_orders['delivery_days']>=0)
]
refund_orders['new_refund_amount'] = refund_orders['shipping_fee']
refund_orders['new_shipping_fee'] = 0

refund_orders.head()


,order_id,customer_id,city,channel,category,order_date,quantity,unit_price,discount_pct,shipping_fee,payment,status,delivery_days,refund_amount,coupon_code,new_shipping_fee,new_discount_pct,new_refund_amount
row_id,,,,,,,,,,,,,,,,,,
1006,ORD0003,190,Delhi,Store,Clothing,2026-07-17,5,3985.0,10.0,49.0,COD,Cancelled,NaN,0.0,NEW00,0,0,49.0
1045,ORD0016,147,Mumbai,Website,Grocery,2026-08-08,4,1472.0,20.0,149.0,Prepaid,Cancelled,NaN,NaN,vip05,0,0,149.0
1108,ORD0037,159,Delhi,App,Clothing,2026-09-05,4,6561.0,5.0,0.0,Prepaid,Cancelled,NaN,0.0,new20,0,0,0.0
1111,ORD0038,184,Bengaluru,App,Clothing,2026-08-29,2,1540.0,0.0,99.0,Prepaid,Cancelled,NaN,0.0,NEW00,0,0,99.0
1174,ORD0059,159,Delhi,Website,Furniture,2026-06-04,4,13741.0,0.0,99.0,Prepaid,Cancelled,NaN,0.0,new20,0,0,99.0


**Question 3**

Create an explicit independent slice named `patch` containing Electronics orders with quantity at least 2 and status Delivered or Returned. Include `order_id`, `status`, `quantity`, `unit_price`, `discount_pct`.

Inside `patch`, add 5 percentage points to Delivered-order discounts, capped at 20%; set Returned-order discounts to 0%. Add `rebate_amount = quantity × unit_price × updated discount_pct / 100`, rounded to two decimals. Sort `patch` by index **descending**.

Write only `discount_pct` back to `orders` by matching index labels, using a DataFrame on the right-hand side. Keep `rebate_amount` only in `patch`, and preserve all non-selected rows. Return an audit table sorted by index ascending: `order_id`, `status`, `old_discount_pct`, `new_discount_pct`, `rebate_amount`.


In [8]:
# Start from the original data; each question is independent.
orders = make_orders()

selected = (
    (orders["category"] == "Electronics")
    & (orders["quantity"] >= 2)
    & orders["status"].isin(["Delivered", "Returned"])
)
patch = orders.loc[
    selected, ["order_id", "status", "quantity", "unit_price", "discount_pct"]
].copy()
old_discount_pct = patch["discount_pct"].copy()

delivered = patch["status"] == "Delivered"
returned = patch["status"] == "Returned"
patch.loc[delivered, "discount_pct"] =+ 5
patch.loc[patch["discount_pct"] > 20, "discount_pct"] = 20
patch.loc[returned, "discount_pct"] = 0
patch["rebate_amount"] = (
    patch["quantity"] * patch["unit_price"] * patch["discount_pct"] / 100
).round(2)
patch = patch.sort_index(ascending=False)

# Write back only the updated discount values, matched by index label.
# print(patch.index)
orders.loc[patch.index, ["discount_pct"]] = patch[["discount_pct"]]

audit = patch.copy()
audit["old_discount_pct"] = old_discount_pct
audit = audit.sort_index().rename(columns={"discount_pct": "new_discount_pct"})
audit


,order_id,status,quantity,unit_price,new_discount_pct,rebate_amount,old_discount_pct
row_id,,,,,,,
1000,ORD0001,Delivered,5,14802.0,5.0,3700.50,20.0
1012,ORD0005,Delivered,6,20013.0,5.0,6003.90,0.0
1036,ORD0013,Delivered,4,20381.0,5.0,4076.20,0.0
1039,ORD0014,Delivered,5,14565.0,5.0,3641.25,20.0
1048,ORD0017,Delivered,4,4045.0,5.0,809.00,0.0
...,...,...,...,...,...,...,...
2398,ORD0467,Delivered,6,13346.0,5.0,4003.80,10.0
2416,ORD0473,Delivered,5,5916.0,5.0,1479.00,5.0
2422,ORD0475,Delivered,5,6541.0,5.0,1635.25,20.0


**Question 4**

Write a named function and apply it to the `coupon_code` **Series** to create `coupon_clean`.

Missing or whitespace-only values become `NO_COUPON`. Otherwise strip surrounding spaces and uppercase the code. A valid code must have exactly three ASCII letters followed by exactly two digits, and the numeric suffix must be from 01 through 50 inclusive. Return the normalized code if valid; otherwise return `INVALID`.

Return `order_id`, `coupon_code`, `coupon_clean`, sorted by index ascending. Leave the raw column unchanged.


In [10]:
orders = make_orders()

def check_code_is_upper(code):
    for char in code:
        if char.isupper():
            continue


def clean_coupon(value):
    if pd.isna(value) or not str(value).strip():
        return "NO_COUPON"
    coupon_code = str(value).strip().upper()
    if check_code_is_upper(coupon_code) and coupon_code.isascii():
        suffix = int(coupon_code[-2:])
        if 1 <= suffix <= 50:
            return coupon_code
    return "INVALID"

orders["coupon_clean"] = orders["coupon_code"].apply(clean_coupon)
coupon_result = orders.loc[:, ["order_id", "coupon_code", "coupon_clean"]].sort_index()
coupon_result


,order_id,coupon_code,coupon_clean
row_id,,,
1000,ORD0001,save10,INVALID
1003,ORD0002,NaN,NO_COUPON
1006,ORD0003,NEW00,INVALID
1009,ORD0004,A1B20,INVALID
1012,ORD0005,VIP75,INVALID
...,...,...,...
2485,ORD0496,A1B20,INVALID
2488,ORD0497,NEW00,INVALID
2491,ORD0498,NEW20,INVALID


**Question 5**

Use a named function with `orders.apply(..., axis=1)` to produce two columns in one call: `taxable_amount` and `invoice_total`.

For Cancelled orders both are 0. For all other statuses, taxable amount is `quantity × unit_price × (1 − discount_pct / 100)`, rounded to two decimals. Invoice total is that rounded taxable amount plus shipping fee minus refund, floored at zero and rounded to two decimals. Treat a missing refund as zero. The function must return a Series with the two named fields; add the result to `orders` by index alignment.

Return `order_id`, `status`, `quantity`, `unit_price`, `discount_pct`, `shipping_fee`, `refund_amount`, `taxable_amount`, `invoice_total`, sorted by index ascending.


In [11]:
orders = make_orders()

def calculate_invoice(row):
    if row["status"] == "Cancelled":
        taxable_amount = 0.0
        invoice_total = 0.0
    else:
        taxable_amount = round(
            row["quantity"] * row["unit_price"] * (1 - row["discount_pct"] / 100), 2
        )
        refund = 0.0 if pd.isna(row["refund_amount"]) else row["refund_amount"]
        invoice_total = round(max(0.0, taxable_amount + row["shipping_fee"] - refund), 2)
    return pd.Series({
        "taxable_amount": taxable_amount,
        "invoice_total": invoice_total,
    })

orders[["taxable_amount", "invoice_total"]] = orders.apply(calculate_invoice, axis=1)
invoice_result = orders.loc[:, [
    "order_id", "status", "quantity", "unit_price", "discount_pct",
    "shipping_fee", "refund_amount", "taxable_amount", "invoice_total",
]].sort_index()
invoice_result


,order_id,status,quantity,unit_price,discount_pct,shipping_fee,refund_amount,taxable_amount,invoice_total
row_id,,,,,,,,,
1000,ORD0001,Delivered,5,14802.0,20.0,49.0,700.0,59208.0,58557.0
1003,ORD0002,Delivered,1,17871.0,20.0,0.0,NaN,14296.8,14296.8
1006,ORD0003,Cancelled,5,3985.0,10.0,49.0,0.0,0.0,0.0
1009,ORD0004,Delivered,6,14406.0,15.0,0.0,700.0,73470.6,72770.6
1012,ORD0005,Delivered,6,20013.0,0.0,49.0,0.0,120078.0,120127.0
...,...,...,...,...,...,...,...,...,...
2485,ORD0496,Delivered,2,575.0,0.0,49.0,NaN,1150.0,1199.0
2488,ORD0497,Delivered,6,581.0,20.0,49.0,700.0,2788.8,2137.8
2491,ORD0498,In transit,2,3514.0,0.0,149.0,0.0,7028.0,7177.0


DataFrame 2: employees

In [9]:
def make_employees():
    rng = np.random.default_rng(202)
    n = 500
    return pd.DataFrame({
        "employee_id": [f"EMP{i:04d}" for i in range(1, n + 1)],
        "department": rng.choice(["Data", "Engineering", "Sales", "Support", "HR"], n),
        "city": rng.choice(["Gurgaon", "Pune", "Hyderabad", "Chennai"], n),
        "grade": rng.choice(["Junior", "Mid", "Senior", "Intern"], n),
        "tenure_months": rng.integers(3, 73, n),
        "monthly_salary": rng.integers(25000, 160001, n).astype(float),
        "performance_rating": rng.choice([2.4, 2.9, 3.3, 3.8, 4.2, 4.5, 4.8, np.nan], n),
        "attendance_pct": rng.choice([85.0, 89.0, 92.0, 95.0, 97.0, 99.0, np.nan], n),
        "overtime_hours": rng.integers(0, 46, n),
        "leave_taken": rng.integers(0, 26, n),
        "leave_allowed": rng.choice([18, 21, 24], n),
        "training_hours": rng.integers(0, 41, n),
        "work_mode": rng.choice(["Office", "Hybrid", "Remote"], n),
        "manager_id": rng.choice(["M01", "M02", "M03", "M04", None], n),
        "skills_text": rng.choice([" Python | SQL | python ", "PY, Excel, SQL", " pandas | py | SQL ", "Excel,excel,Power BI", " postgres | SQL | Python ", " , | ", None, "Git | Docker, git"], n),
        "employment_status": rng.choice(["Active", "Notice", "Inactive"], n, p=[0.8, 0.12, 0.08]),
        "variable_pay": np.zeros(n),
    }, index=pd.Index(2000 + np.arange(n) * 5, name="row_id"))

employees = make_employees()

In [15]:
employees.index

Index([2000, 2005, 2010, 2015, 2020, 2025, 2030, 2035, 2040, 2045,
       ...
       4450, 4455, 4460, 4465, 4470, 4475, 4480, 4485, 4490, 4495],
      dtype='int64', name='row_id', length=500)

In [16]:
employees.head()

,employee_id,department,city,grade,tenure_months,monthly_salary,performance_rating,attendance_pct,overtime_hours,leave_taken,leave_allowed,training_hours,work_mode,manager_id,skills_text,employment_status,variable_pay
row_id,,,,,,,,,,,,,,,,,
2000,EMP0001,Data,Hyderabad,Senior,4,140062.0,4.8,95.0,2,10,24,9,Hybrid,M02,"PY, Excel, SQL",Active,0.0
2005,EMP0002,Engineering,Pune,Senior,69,49868.0,2.9,92.0,8,15,24,12,Office,M03,"Excel,excel,Power BI",Active,0.0
2010,EMP0003,Sales,Chennai,Senior,68,89938.0,3.8,95.0,33,3,18,21,Remote,M03,"PY, Excel, SQL",Active,0.0
2015,EMP0004,Sales,Pune,Senior,31,78558.0,2.4,89.0,17,24,18,39,Remote,M03,"Excel,excel,Power BI",Active,0.0
2020,EMP0005,Engineering,Chennai,Junior,20,70724.0,NaN,95.0,38,16,18,30,Remote,M04,pandas | py | SQL,Active,0.0


### Question 6



---


In [ ]:
# Start from the original data for Question 6.
employees = make_employees()

employee_slice = employees.loc[2050:2400]
selected = (
    employee_slice["employment_status"].eq("Active")
    & employee_slice["department"].isin(["Data", "Engineering", "Support"])
    & employee_slice["grade"].ne("Intern")
    & employee_slice["performance_rating"].ge(3.3)
    & (
        employee_slice["attendance_pct"].lt(92)
        | employee_slice["overtime_hours"].gt(24)
    )
)
result_q6 = employee_slice.loc[
    selected,
    ["employee_id", "department", "grade", "performance_rating", "attendance_pct", "overtime_hours"],
].sort_index()
result_q6


Question 7


In [ ]:
employees = make_employees()
old_salary = employees["monthly_salary"].copy()

eligible = (
    employees["employment_status"].eq("Active")
    & employees["grade"].isin(["Junior", "Mid", "Senior"])
    & employees["performance_rating"].ge(4.2)
    & employees["attendance_pct"].ge(95)
    & employees["leave_taken"].le(employees["leave_allowed"])
    & employees["overtime_hours"].le(30)
)
data_engineering = eligible & employees["department"].isin(["Data", "Engineering"])
other_departments = eligible & ~employees["department"].isin(["Data", "Engineering"])

employees.loc[data_engineering, "monthly_salary"] = (old_salary[data_engineering] * 1.07).round(2)
employees.loc[other_departments, "monthly_salary"] = (old_salary[other_departments] * 1.04).round(2)

junior = eligible & employees["grade"].eq("Junior")
mid_senior = eligible & employees["grade"].isin(["Mid", "Senior"])
employees.loc[junior, "variable_pay"] = (old_salary[junior] * 12 * 0.05).round(2)
employees.loc[mid_senior, "variable_pay"] = (old_salary[mid_senior] * 12 * 0.08).round(2)

result_q7 = employees.loc[eligible, ["employee_id", "department", "grade"]].copy()
result_q7["old_monthly_salary"] = old_salary[eligible]
result_q7["new_monthly_salary"] = employees.loc[eligible, "monthly_salary"]
result_q7["variable_pay"] = employees.loc[eligible, "variable_pay"]
result_q7 = result_q7.sort_index()
result_q7


Question 8


In [ ]:
# Start from the original data for Question 8.
employees = make_employees()
selected = employees["employment_status"].eq("Active") & employees["overtime_hours"].ge(25)
patch = employees.loc[selected, ["employee_id", "overtime_hours", "training_hours", "performance_rating"]].copy()
old_overtime = patch["overtime_hours"].copy()
old_training = patch["training_hours"].copy()

patch["overtime_hours"] = patch["overtime_hours"] - 5
low_or_missing_rating = patch["performance_rating"].isna() | (patch["performance_rating"] < 3.5)
patch.loc[low_or_missing_rating, "training_hours"] = patch.loc[low_or_missing_rating, "training_hours"] + 4
patch.loc[~low_or_missing_rating, "training_hours"] = patch.loc[~low_or_missing_rating, "training_hours"] + 2
patch = patch.sort_index(ascending=False)

# The DataFrame columns are intentionally in the opposite order.
employees.loc[patch.index, ["overtime_hours", "training_hours"]] = patch[["training_hours", "overtime_hours"]]

result_q8 = patch[["employee_id"]].copy()
result_q8["old_overtime_hours"] = old_overtime
result_q8["new_overtime_hours"] = patch["overtime_hours"]
result_q8["old_training_hours"] = old_training
result_q8["new_training_hours"] = patch["training_hours"]
result_q8 = result_q8.sort_index()
result_q8


Question 9


In [ ]:
employees = make_employees()

def clean_skills(value):
    if pd.isna(value):
        return "UNKNOWN"
    tokens = []
    for token in re.split(r"[,|]", value):
        token = token.strip().lower()
        if token:
            if token == "py":
                token = "python"
            elif token == "postgres":
                token = "sql"
            tokens.append(token)
    tokens = sorted(set(tokens))
    if not tokens:
        return "UNKNOWN"
    return " | ".join(tokens)

employees["skills_clean"] = employees["skills_text"].apply(clean_skills)
result_q9 = employees.loc[:, ["employee_id", "skills_text", "skills_clean"]].sort_index()
result_q9


Question 10

In [ ]:
# Start from the original data for Question 10.
employees = make_employees()

def employee_decision(row):
    if row["employment_status"] != "Active":
        decision, reason = "NOT_ELIGIBLE", "STATUS"
    elif pd.isna(row["manager_id"]):
        decision, reason = "MANUAL_REVIEW", "NO_MANAGER"
    elif pd.isna(row["performance_rating"]) or pd.isna(row["attendance_pct"]):
        decision, reason = "INSUFFICIENT_DATA", "MISSING_METRIC"
    elif row["grade"] == "Senior":
        decision, reason = "NO_OPEN_GRADE", "SENIOR"
    elif row["performance_rating"] < 3 or row["attendance_pct"] < 90:
        decision, reason = "PERFORMANCE_PLAN", "LOW_METRIC"
    elif (
        row["grade"] in ["Junior", "Mid"]
        and row["tenure_months"] >= 24
        and row["performance_rating"] >= 4.4
        and row["attendance_pct"] >= 96
        and row["training_hours"] >= 20
        and row["leave_taken"] <= row["leave_allowed"]
    ):
        decision, reason = "PROMOTE", "ALL_CRITERIA"
    else:
        decision, reason = "DEVELOP", "CRITERIA_NOT_MET"
    return pd.Series({"decision": decision, "reason": reason})

employees[["decision", "reason"]] = employees.apply(employee_decision, axis=1)
result_q10 = employees.loc[:, [
    "employee_id", "employment_status", "manager_id", "grade",
    "performance_rating", "attendance_pct", "decision", "reason",
]].sort_index()
result_q10
